In [77]:
import requests
import pandas as pd
import json
import numpy as np
from datetime import datetime, timezone
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas
import os
import hashlib
from dotenv import load_dotenv

RECUPERATION DES DIM CITY SNOWFLAKE

In [78]:
ACOUNT_SNOWFLAKE = os.getenv('ACOUNT_SNOWFLAKE')
USER_SNOWFLAKE = os.getenv('USER_SNOWFLAKE')
PASSWORD_SNOWFLAKE = os.getenv('PASSWORD_SNOWFLAKE')
conn = snowflake.connector.connect(
    user=USER_SNOWFLAKE,
    password=PASSWORD_SNOWFLAKE,
    account=ACOUNT_SNOWFLAKE,  
    warehouse="COMPUTE_WH",
    database="GOOD_AIR",
    schema="SILVER"
)

# --- ETAPE 1 : RECUPERATION ---
print("Récupération des données...")
query = "SELECT * FROM DIM_CITY"
df_dim_city = pd.read_sql(query, conn)

Récupération des données...


C:\Users\Utilisateur\AppData\Local\Temp\ipykernel_1640\647133777.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_dim_city = pd.read_sql(query, conn)


TRAITEMENT DE LA DONNEE HISTORIQUE

In [79]:
#boucle sur tout les csv et concat
path_hist = r"../historique_AQI"
list_csv = os.listdir(path_hist)
list_csv
list_df = []

for city in list_csv :
    df_city = pd.read_csv(path_hist+f'/{city}', sep=', ')
    df_city['CITY_NAME'] = city.replace('.csv','')
    list_df.append(df_city)

df_citys = pd.concat(list_df)


C:\Users\Utilisateur\AppData\Local\Temp\ipykernel_1640\2610180832.py:8: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  df_city = pd.read_csv(path_hist+f'/{city}', sep=', ')
C:\Users\Utilisateur\AppData\Local\Temp\ipykernel_1640\2610180832.py:8: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  df_city = pd.read_csv(path_hist+f'/{city}', sep=', ')
C:\Users\Utilisateur\AppData\Local\Temp\ipykernel_1640\2610180832.py:8: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid

In [80]:
print("Dossier actuel :", os.getcwd())

Dossier actuel : d:\DATA\2025-11-28_MSPR-1_2\Good-Air\etl


In [81]:
#Ajout des id
df_citys = df_citys.merge(df_dim_city,on='CITY_NAME', how='left')

#Nettoyage nom de colonne
df_citys = df_citys[['CITY_ID','CITY_NAME','date','pm25','pm10','o3']]
df_citys.rename(columns={'date':'DT_PARIS','pm10':'IAQI_PM10', 'pm25':'IAQI_PM25','o3':'IAQI_O3'}, inplace=True)

#conversion des colonne feature en colonne numerique 
cols = ['IAQI_PM10', 'IAQI_PM25', 'IAQI_O3']
df_citys[cols] = df_citys[cols].apply(pd.to_numeric, errors='coerce')

#Ajout de l'IAQI
df_citys["AQI"] = df_citys[cols].max(axis=1)

In [82]:
df = df_citys.copy()

In [83]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72869 entries, 0 to 72868
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   CITY_ID    72869 non-null  object 
 1   CITY_NAME  72869 non-null  object 
 2   DT_PARIS   72869 non-null  object 
 3   IAQI_PM25  58187 non-null  float64
 4   IAQI_PM10  67483 non-null  float64
 5   IAQI_O3    55623 non-null  float64
 6   AQI        72342 non-null  float64
dtypes: float64(4), object(3)
memory usage: 3.9+ MB


In [84]:
#Transformation de la date 
df['DT_PARIS'] = pd.to_datetime(df["DT_PARIS"])

#Triage du df par date ASC
df = df.sort_values(by=['CITY_ID', 'DT_PARIS'])

In [85]:
#Remplissage 
def interpoler_aqi(df, colonnes_polluants):
    # On s'assure que les colonnes sont bien numériques avant de commencer
    df[colonnes_polluants] = df[colonnes_polluants].apply(pd.to_numeric, errors='coerce')
    
    # On trie par ville et par date pour que "précédent/suivant" ait un sens
    df = df.sort_values(by=['CITY_ID', 'DT_PARIS'])

    def remplir_groupe(group):
        for col in colonnes_polluants:
            # .shift(1) : ligne précédente | .shift(-1) : ligne suivante
            prev_val = group[col].shift(1)
            next_val = group[col].shift(-1)
            
            # Cas 1 : Les deux sont présentes -> Moyenne
            mask_both = group[col].isna() & prev_val.notna() & next_val.notna()
            group.loc[mask_both, col] = (prev_val + next_val) / 2
            
            # Cas 2 : Seule la précédente est présente
            mask_prev = group[col].isna() & prev_val.notna()
            group.loc[mask_prev, col] = prev_val
            
            # Cas 3 : Seule la suivante est présente
            mask_next = group[col].isna() & next_val.notna()
            group.loc[mask_next, col] = next_val
            
        return group

    # 1. Appliquer l'interpolation par ville
    df = df.groupby('CITY_ID', group_keys=False).apply(remplir_groupe)
    
    # 2. Remplacer les NaN résiduels (ceux qui n'ont pu être interpolés) par -999
    df[colonnes_polluants] = df[colonnes_polluants].fillna(-999)
    
    return df

# Utilisation :
df = interpoler_aqi(df, cols)

C:\Users\Utilisateur\AppData\Local\Temp\ipykernel_1640\659180182.py:30: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('CITY_ID', group_keys=False).apply(remplir_groupe)


In [86]:
# 1. Normaliser la date au format Snowflake (avec l'heure à 00:00:00.000)
# On passe de '2014-08-24' à '2014-08-24 00:00:00.000'
df['DT_PARIS'] = pd.to_datetime(df['DT_PARIS']).dt.strftime('%Y-%m-%d %H:%M:%S.000')

# 2. Fonction de hachage identique à Snowflake (SHA256 au vu de ta clé exemple)
def generate_snowflake_id(row):
    # Concaténation : CITY_NAME + '-' + DATE_STR
    combined = f"{row['CITY_NAME']}-{row['DT_PARIS']}"
    
    # Encodage en UTF-8 puis SHA256 (plus probable que MD5 vu la longueur)
    # .upper() car Snowflake renvoie les hashs en majuscules par défaut
    return hashlib.md5(combined.encode('utf-8')).hexdigest().upper()

# 3. Application
df['RECORD_ID'] = df.apply(generate_snowflake_id, axis=1)

# Nettoyage de la colonne temporaire
df.drop(columns=['CITY_NAME'], inplace=True)

Export dans snowflake

In [87]:
# 3. Envoi massif (One-shot)
success, nchunks, nrows, _ = write_pandas(
    conn=conn,
    df=df,
    table_name='FACT_AIR_QUALITY_RECORDS', # Le nom de la table dans Snowflake
    database='GOOD_AIR',
    schema='SILVER',
    quote_identifiers=False # Permet de ne pas être sensible à la casse si la table est déjà créée en majuscules
)

if success:
    print(f"Succès ! {nrows} lignes ont été insérées dans Snowflake.")

# N'oublie pas de fermer la connexion si tu n'en as plus besoin
# conn.close()

C:\Users\Utilisateur\AppData\Local\Temp\ipykernel_1640\3068587469.py:2: UserWarning: Pandas Dataframe has non-standard index of type <class 'pandas.core.indexes.base.Index'> which will not be written. Consider changing the index to pd.RangeIndex(start=0,...,step=1) or call reset_index() to keep index as column(s)
  success, nchunks, nrows, _ = write_pandas(


Succès ! 72869 lignes ont été insérées dans Snowflake.


In [89]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 72869 entries, 13265 to 17735
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   CITY_ID    72869 non-null  object 
 1   DT_PARIS   72869 non-null  object 
 2   IAQI_PM25  72869 non-null  float64
 3   IAQI_PM10  72869 non-null  float64
 4   IAQI_O3    72869 non-null  float64
 5   AQI        72342 non-null  float64
 6   RECORD_ID  72869 non-null  object 
dtypes: float64(4), object(3)
memory usage: 4.4+ MB
